source : https://mbrenndoerfer.com/writing/hierarchical-clustering-complete-guide-dendrograms-linkage-criteria#implementation-in-scikit-learn

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score

## Generate and Prepare Sample DataLink
We'll create a synthetic dataset with known cluster structure to evaluate our clustering performance:

In [21]:
# Generate sample data with 4 distinct clusters
np.random.seed(42)
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=1.5, random_state=42)

# Standardize the data - crucial for clustering algorithms
X = StandardScaler().fit_transform(X)

The dataset contains 300 data points with 2 features, organized into 4 true clusters. The shape (300, 2) indicates we have 300 samples, each with 2 features, which is well-suited for visualization and understanding how hierarchical clustering works. Standardization ensures that both features contribute equally to the distance calculations, preventing features with larger scales from dominating the clustering process.

## Apply Hierarchical Clustering with Different Linkage CriteriaLink
Now we'll compare different linkage criteria to see how they affect clustering results:

In [16]:
# Define linkage methods to compare
linkage_methods = ['single', 'complete', 'average', 'ward']
clustering_results = {}

# Apply hierarchical clustering with each linkage method
for method in linkage_methods:
    # Note: ward linkage only works with euclidean distance
    clustering = AgglomerativeClustering(
        n_clusters=4,           # Number of clusters to find
        linkage=method,         # Linkage criterion
        metric='euclidean'      # Distance metric
    )

    # Fit the model and predict cluster labels
    y_pred = clustering.fit_predict(X)
    clustering_results[method] = y_pred

## Evaluate clustering performance

In [17]:
# Calculate performance metrics for each method
metrics = {}
for method, y_pred in clustering_results.items():
    ari = adjusted_rand_score(y_true, y_pred)
    silhouette = silhouette_score(X, y_pred)
    metrics[method] = {'ari': ari, 'silhouette': silhouette}

for m in metrics:
    print(f"{m} linkage:")
    print(f"  Adjusted Rand Index: {round(metrics[m]['ari'], 3)}")
    print(f"  Silhouette Score: {round(metrics[m]['silhouette'], 3)}")

single linkage:
  Adjusted Rand Index: 0.333
  Silhouette Score: 0.259
complete linkage:
  Adjusted Rand Index: 0.951
  Silhouette Score: 0.682
average linkage:
  Adjusted Rand Index: 0.973
  Silhouette Score: 0.689
ward linkage:
  Adjusted Rand Index: 0.974
  Silhouette Score: 0.69


The results demonstrate how different linkage criteria perform on our dataset. The Adjusted Rand Index (ARI) measures how well the clustering matches the true labels, with values ranging from -1 to 1. An ARI of 1 indicates complete agreement with the true clusters, while values closer to 0 suggest random clustering. The Silhouette Score measures cluster quality based on how well-separated clusters are, with values ranging from -1 to 1. Values closer to 1 indicate well-separated, cohesive clusters, while values below 0.2 suggest clusters may not be well-separated.

In this example, average linkage and Ward linkage typically achieve the best balance, with ARI scores above 0.9 and silhouette scores above 0.5. These high scores indicate strong agreement with the true cluster structure and well-separated clusters. Single linkage often performs poorly on this type of data due to its tendency to create chain-like clusters, while complete linkage may be overly strict and break up natural clusters. The performance differences highlight the importance of choosing the appropriate linkage criterion based on your data characteristics and clustering goals.

In [24]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

# Generate the linkage matrix using average linkage
linkage_matrix = linkage(X, method='average')

# Extract clusters at different levels by specifying the number of clusters
clusters_2 = fcluster(linkage_matrix, 2, criterion='maxclust')
clusters_4 = fcluster(linkage_matrix, 4, criterion='maxclust')

# Count points in each cluster for both clusterings
cluster_counts_2 = np.unique(clusters_2, return_counts=True)
cluster_counts_4 = np.unique(clusters_4, return_counts=True)

print(cluster_counts_2)
print(cluster_counts_4)

(array([1, 2], dtype=int32), array([ 75, 225]))
(array([1, 2, 3, 4], dtype=int32), array([75, 75, 75, 75]))
